# Chapter 16 &mdash; Two Equivalent Definitions of NP-Completeness

**Concept 7 of the Chapter 16 decomposition:** *Two Equivalent Definitions of NP-Completeness*

(a) in NP and every NP language reduces to it; (b) in NP and some known NPC problem reduces to it.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Two-Definitions-Of-NPC/Concept-Two-Definitions-Of-NPC.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Two definitions, one used for the **first** NPC problem and one for every problem
after it.

**(a) From first principles.** $L$ is NPC if $L\in NP$ and **every** $A\in NP$
satisfies $A \le_p L$. This is what Cook and Levin proved for SAT &mdash; and it requires
reasoning about an arbitrary NP machine.

**(b) By reduction from a known NPC problem.** $L$ is NPC if $L\in NP$ and $B \le_p L$
for some **known** NPC $B$. Since $\le_p$ is transitive, every NP problem reaches $L$
through $B$.

So (a) is used **once** and (b) ever after. The recipe for proving $L$ NPC:

1. show $L \in NP$ &mdash; exhibit a certificate and a polynomial verifier;
2. pick a known NPC $B$;
3. give a **polynomial-time** $f$ with $x\in B \iff f(x)\in L$.

Step 1 is the one people forget, and skipping it proves only NP-**hardness**.

## 2. Definitions

### Polynomial-time reducibility, and its transitivity

In [ ]:
def compose(f, g): return lambda x: g(f(x))

REDUCTIONS = {}      # (from, to) -> f
def add_reduction(a, b, f): REDUCTIONS[(a, b)] = f

def reduces_to(a, b, seen=None):
    seen = seen or set()
    if (a, b) in REDUCTIONS: return True
    for (x, y) in REDUCTIONS:
        if x == a and y not in seen and reduces_to(y, b, seen | {a}):
            return True
    return False

### A concrete chain of reductions

In [ ]:
# --- a tiny CNF toolkit -------------------------------------------------
# A literal is an int: 3 means x3, -3 means NOT x3.
# A clause is a tuple of literals; a formula is a list of clauses.
from itertools import product

def nvars(F):
    return max((abs(l) for c in F for l in c), default=0)

def evaluate(F, assign):
    # assign: dict var -> bool
    return all(any(assign[abs(l)] == (l > 0) for l in c) for c in F)

def brute_sat(F):
    n = nvars(F)
    for bits in product([False, True], repeat=n):
        a = {i + 1: bits[i] for i in range(n)}
        if evaluate(F, a): return a
    return None

def show_cnf(F):
    def lit(l): return ("x%d" % l) if l > 0 else ("~x%d" % -l)
    return " AND ".join("(" + " OR ".join(lit(l) for l in c) + ")" for c in F)


def sat_to_3sat(F, nextvar=None):
    # pad short clauses with fresh variables; split long ones
    n = nextvar or nvars(F)
    out = []
    for c_ in F:
        c_ = list(c_)
        if len(c_) == 1:
            n += 1; p = n; n += 1; q = n
            out += [(c_[0], p, q), (c_[0], p, -q), (c_[0], -p, q), (c_[0], -p, -q)]
        elif len(c_) == 2:
            n += 1; p = n
            out += [(c_[0], c_[1], p), (c_[0], c_[1], -p)]
        elif len(c_) == 3:
            out.append(tuple(c_))
        else:
            while len(c_) > 3:
                n += 1; p = n
                out.append((c_[0], c_[1], p))
                c_ = [-p] + c_[2:]
            out.append(tuple(c_))
    return out

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;6.&nbsp;2-SAT in Polynomial Time: the Implication Graph](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-2SAT-In-Polynomial-Time/Concept-2SAT-In-Polynomial-Time.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16-NPC/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;8.&nbsp;The Cook–Levin Theorem: 3-SAT is NP-Complete](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Cook-Levin/Concept-Cook-Levin.ipynb)&nbsp;&rarr;

---

## 3. Tests

**Definition (b) in action:** SAT $\le_p$ 3-SAT.

In [ ]:
F = [(1,), (2, 3), (1, -2, 3, 4), (-1, -3)]
print("original :", show_cnf(F))
G = sat_to_3sat(F)
print("3-CNF    :", show_cnf(G)[:100], "...")
print("clauses %d -> %d, variables %d -> %d"
      % (len(F), len(G), nvars(F), nvars(G)))
assert all(len(c_) == 3 for c_ in G)

The reduction preserves satisfiability &mdash; which is all it must do.

In [ ]:
import random
def random_cnf(nv, nc, seed):
    random.seed(seed)
    out = []
    for _ in range(nc):
        k = random.choice([1, 2, 3, 4])
        vs = random.sample(range(1, nv + 1), min(k, nv))
        out.append(tuple(v * random.choice([1, -1]) for v in vs))
    return out

bad = 0
for s in range(40):
    F = random_cnf(4, 5, s)
    G = sat_to_3sat(F)
    if (brute_sat(F) is not None) != (brute_sat(G) is not None): bad += 1
print("40 random formulas : %d satisfiability mismatches" % bad)
assert bad == 0

It is **polynomial**: the output grows linearly in the input.

In [ ]:
print("%-10s %-12s %s" % ("in clauses", "out clauses", "ratio"))
for nc in [5, 20, 80]:
    F = random_cnf(8, nc, 1)
    G = sat_to_3sat(F)
    print("%-10d %-12d %.2f" % (len(F), len(G), len(G) / float(len(F))))

**Transitivity** is what makes definition (b) legitimate.

In [ ]:
add_reduction('SAT', '3-SAT', sat_to_3sat)
add_reduction('3-SAT', 'clique', lambda x: x)
add_reduction('clique', 'vertex cover', lambda x: x)
for a, b in [('SAT', 'clique'), ('SAT', 'vertex cover'), ('3-SAT', 'vertex cover')]:
    print("  %-8s <=p %-14s ? %s" % (a, b, reduces_to(a, b)))
    assert reduces_to(a, b)
print("\nEvery NP problem reaches vertex cover through SAT and 3-SAT.")

**The recipe**, and the step people skip.

In [ ]:
print("to prove L is NP-COMPLETE:")
print("  1. L is in NP          -- certificate + polynomial verifier")
print("  2. pick a known NPC B")
print("  3. give polynomial f with  x in B  <=>  f(x) in L")
print()
print("Skipping step 1 proves only NP-HARDNESS -- and Concept 10 shows")
print("that NP-hard alone can mean UNDECIDABLE.")

## 4. Exercises


1. Which definition did Cook and Levin use, and why did they have to?
2. Prove $\le_p$ is transitive, being careful about the polynomial bound.
3. Where exactly does the 4-literal clause split preserve satisfiability?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 253 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter16-NPC/Concept-Two-Definitions-Of-NPC')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')